In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tqdm import tqdm

In [ ]:
!rm -rf /kaggle/working/tfrecords

In [ ]:
CHOLEC_BASE_DIR = '/kaggle/input/datasets/ganumatta/cholec80'
OPTICAL_FLOW_BASE_DIR = '/kaggle/input/datasets/ganumatta/cholec80-optical-flow/optical_flow'
TFRECORD_DIR = '/kaggle/working/tfrecords'

os.makedirs(TFRECORD_DIR, exist_ok=True)

In [ ]:
def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

phase_map = {
    'Preparation': 0, 'CalotTriangleDissection': 1, 'ClippingCutting': 2,
    'GallbladderDissection': 3, 'GallbladderPackaging': 4,
    'CleaningCoagulation': 5, 'GallbladderRetraction': 6               
} # again should be retrieval but whatever

inverse_phase_map = {v: k for k, v in phase_map.items()} # this instead of just inverted map because the phases aren't encoded in the text file

In [ ]:
video = 'video09'

rgb_dir = os.path.join(CHOLEC_BASE_DIR, 'frames', video)
flow_dir = os.path.join(OPTICAL_FLOW_BASE_DIR, video)
phase_file = os.path.join(CHOLEC_BASE_DIR, 'phase_annotations', f"{video}-phase.txt")

with open(phase_file, 'r') as f:
    raw_phases = [line.strip().split()[-1] for line in f.readlines() if line.strip()]
    if raw_phases[0].lower() == 'phase':
        raw_phases = raw_phases[1:]
    phases = [phase_map[p] for p in raw_phases] # making a list of all the phases in the video and encoding them

frames = sorted(os.listdir(rgb_dir))
record_file = os.path.join(TFRECORD_DIR, f"{video}.tfrecord")

In [ ]:
with tf.io.TFRecordWriter(record_file) as writer:
    for i, frame_name in enumerate(tqdm(frames, desc="Compressing Frames")):
        label_index = min(i * 25, len(phases) - 1) # since there are 25 phase annotations for each frame we just multiply the index by 25 and pack them together
        label = phases[label_index]
        
        rgb_path = os.path.join(rgb_dir, frame_name)
        rgb_img = cv2.imread(rgb_path)
        rgb_img = cv2.resize(rgb_img, (224, 224))
        _, rgb_encoded = cv2.imencode('.jpg', rgb_img, [int(cv2.IMWRITE_JPEG_QUALITY), 85]) # change this to 95 for much better quality. i just ran outta space
        rgb_bytes = rgb_encoded.tobytes()
            
        flow_path = os.path.join(flow_dir, frame_name.replace('.png', '.jpg'))
        if os.path.exists(flow_path):
            flow_img = cv2.imread(flow_path)
            flow_img = cv2.resize(flow_img, (224, 224))
            _, flow_encoded = cv2.imencode('.jpg', flow_img, [int(cv2.IMWRITE_JPEG_QUALITY), 85]) # change this to 95 for much better quality. i just ran outta space
            flow_bytes = flow_encoded.tobytes()
        else:
            flow_bytes = b'' # the 1st frame doesn't have one
        
        feature = {
            'rgb_image': _bytes_feature(rgb_bytes),
            'flow_image': _bytes_feature(flow_bytes),
            'label': _int64_feature(label)
        }
        
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        writer.write(example_proto.SerializeToString())

print(f"\n{video} Successfully Compiled!")

In [ ]:
def parse_tfrecord_fn(example):
    feature_description = {
        'rgb_image': tf.io.FixedLenFeature([], tf.string),
        'flow_image': tf.io.FixedLenFeature([], tf.string),
        'label': tf.io.FixedLenFeature([], tf.int64),
    }
    return tf.io.parse_single_example(example, feature_description)

raw_dataset = tf.data.TFRecordDataset(record_file)
parsed_dataset = raw_dataset.map(parse_tfrecord_fn)

all_labels_extracted = []
for parsed_record in parsed_dataset:
    all_labels_extracted.append(parsed_record['label'].numpy())

total_frames = len(all_labels_extracted)
print(f"Total Frames Inside Record: {total_frames}")

# samples
print("\nFirst 10 frames:")
print([inverse_phase_map[l] for l in all_labels_extracted[:10]])

print(f"\nMiddle 10 frames:")
print([inverse_phase_map[l] for l in all_labels_extracted[total_frames//2 : total_frames//2 + 10]])

print("\nLast 10 frames:")
print([inverse_phase_map[l] for l in all_labels_extracted[-10:]])

In [ ]:
dataset = tf.data.TFRecordDataset(record_file)
dataset = dataset.map(parse_tfrecord_fn['label'])

label_counts = {i: 0 for i in range(7)}
total_frames = 0

for label in dataset:
    label_counts[label.numpy()] += 1
    total_frames += 1

if total_frames == 0: # checking the frame distributions within the tfrecord
    print("Error: TFRecord is empty")
else:
    for phase_id in range(7):
        count = label_counts[phase_id]
        percentage = (count / total_frames) * 100
        print(f"Phase {phase_id} [{phase_names[phase_id][:25]:<25}]: {count:>5,} frames ({percentage:>5.2f}%)")

In [ ]:
all_videos = [f'video{i:02d}' for i in range(1, 81)] # that was just for 1 video now we run the loop for the whole dataset

for video in tqdm(all_videos, desc="Overall Progress"):
    rgb_dir = os.path.join(CHOLEC_BASE_DIR, 'frames', video)
    flow_dir = os.path.join(OPTICAL_FLOW_BASE_DIR, video)
    phase_file = os.path.join(CHOLEC_BASE_DIR, 'phase_annotations', f"{video}-phase.txt")
    
    if not os.path.exists(rgb_dir) or not os.path.exists(phase_file):
        print(f"\nSkipping {video} - Data not found.")
        continue

    with open(phase_file, 'r') as f:
        raw_phases = [line.strip().split()[-1] for line in f.readlines() if line.strip()]
        if raw_phases[0].lower() == 'phase':
            raw_phases = raw_phases[1:]
        phases = [phase_map[p] for p in raw_phases]

    frames = sorted(os.listdir(rgb_dir))
    record_file = os.path.join(TFRECORD_DIR, f"{video}.tfrecord")

    with tf.io.TFRecordWriter(record_file) as writer:
        for i, frame_name in enumerate(tqdm(frames, desc=f"Compressing {video}", leave=False)):
            
            label_index = min(i * 25, len(phases) - 1)
            label = phases[label_index]
            
            rgb_path = os.path.join(rgb_dir, frame_name)
            rgb_img = cv2.imread(rgb_path)
            rgb_img = cv2.resize(rgb_img, (224, 224))
            _, rgb_encoded = cv2.imencode('.jpg', rgb_img, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
            rgb_bytes = rgb_encoded.tobytes()
                
            flow_path = os.path.join(flow_dir, frame_name.replace('.png', '.jpg'))
            if os.path.exists(flow_path):
                flow_img = cv2.imread(flow_path)
                flow_img = cv2.resize(flow_img, (224, 224))
                _, flow_encoded = cv2.imencode('.jpg', flow_img, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
                flow_bytes = flow_encoded.tobytes()
            else:
                flow_bytes = b''
            
            feature = {
                'rgb_image': _bytes_feature(rgb_bytes),
                'flow_image': _bytes_feature(flow_bytes),
                'label': _int64_feature(label)
            }
            
            example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
            writer.write(example_proto.SerializeToString())

print("All 80 TFRecords compiled into /kaggle/working/tfrecords/")